In [3]:
!pip install openai chromadb python-dotenv

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

In [11]:
from google.colab import userdata
load_dotenv()

# Get the API key from environment variables
openai_api_key = userdata.get("GOOGLE_API_KEY")

In [12]:
# --------------------------------------------------
# 1. ENV + CLIENT SETUP
# --------------------------------------------------
load_dotenv()

client = OpenAI(api_key=openai_api_key)

In [13]:

# --------------------------------------------------
# 2. EMBEDDING FUNCTION
# --------------------------------------------------
def get_embedding(text: str) -> list[float]:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding



In [14]:
# --------------------------------------------------
# 3. VECTOR DATABASE SETUP (Chroma)
# --------------------------------------------------
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="documents"
)

documents = [
    "LangChain is an orchestration framework for large language models.",
    "RAG stands for Retrieval Augmented Generation.",
    "Vector databases store embeddings for similarity search.",
    "LLMs cannot access private data unless it is provided in the prompt."
]

for idx, doc in enumerate(documents):
    collection.add(
        ids=[str(idx)],
        documents=[doc],
        embeddings=[get_embedding(doc)]
    )

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: AIzaSyBR***************************FlJ8. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [10]:
# --------------------------------------------------
# 4. RETRIEVAL FUNCTION
# --------------------------------------------------
def retrieve_context(query: str, k: int = 2) -> str:
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return "\n".join(results["documents"][0])


# --------------------------------------------------
# 5. LLM ANSWER FUNCTION
# --------------------------------------------------
def answer_question(question: str) -> str:
    context = retrieve_context(question)

    prompt = f"""
You are an expert assistant.
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content


# --------------------------------------------------
# 6. RUN
# --------------------------------------------------
if __name__ == "__main__":
    question = "What is LangChain?"
    answer = answer_question(question)
    print("\nANSWER:\n", answer)


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: de2a2b65************************9f39. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}